# 05 — Validation and Sensitivity

**Goal:** test how valuation and calibration change under simple rate, volatility and model-parameter scenarios. The purpose is model understanding, not a full production Greeks framework.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import QuantLib as ql

from src.rates_project import *
set_evaluation_date()
print("QuantLib version:", ql.__version__)
print("Evaluation date:", ql.Settings.instance().evaluationDate)

QuantLib version: 1.43
Evaluation date: January 15th, 2026


In [2]:
# Base case
base_curve, base_handle, _ = build_curve()
base_helpers, base_meta = build_swaption_helpers(base_handle)
base_model = calibrate_hull_white(base_handle, base_helpers)
base_a, base_sigma = [float(x) for x in base_model.params()]
base_swaption, _, base_atm, _ = make_european_swaption(base_handle, 2, 5, 1_000_000)
base_price = price_swaption_hw(base_swaption, base_model)
base_calib = calibration_table(base_model, base_meta)

print("Base a:", base_a)
print("Base sigma:", base_sigma)
print("Base 2Yx5Y ATM price:", base_price)

Base a: 5.0660325050970636e-05
Base sigma: 0.005638537534606625
Base 2Yx5Y ATM price: 14435.12374813855


## 1. Parallel curve bumps

We rebuild and recalibrate the model after a ±10 bp parallel shift to the input rate quotes. To make the comparison meaningful, we keep the **base ATM strike** fixed when creating the stressed swaptions.

In [3]:
rate_rows = []
for label, shift in [("-10bp", -0.0010), ("Base", 0.0), ("+10bp", 0.0010)]:
    curve, handle, _ = build_curve(rate_shift=shift)
    helpers, meta = build_swaption_helpers(handle)
    model = calibrate_hull_white(handle, helpers)
    swpt, _, stressed_atm, _ = make_european_swaption(handle, 2, 5, 1_000_000, strike=base_atm)
    price = price_swaption_hw(swpt, model)
    a, sigma = [float(x) for x in model.params()]
    rate_rows.append({
        "scenario": label, "curve_shift": shift,
        "stressed_atm_rate": stressed_atm,
        "a": a, "sigma": sigma, "NPV": price,
        "NPV_change": price - base_price,
    })
rate_sensitivity = pd.DataFrame(rate_rows)
rate_sensitivity

,scenario,curve_shift,stressed_atm_rate,a,sigma,NPV,NPV_change
0,-10bp,-0.001,0.025157,0.000027,0.005432,11848.738267,-2586.385481
1,Base,0.000,0.026160,0.000051,0.005639,14435.123748,0.000000
2,+10bp,0.001,0.027164,0.000015,0.005860,17253.947938,2818.824190


## 2. Swaption-volatility bumps

We bump every quoted swaption volatility by ±1 volatility point, recalibrate Hull–White and reprice the same base-strike swaption.

In [4]:
vol_rows = []
for label, vol_shift in [("-1 vol pt", -0.01), ("Base", 0.0), ("+1 vol pt", 0.01)]:
    helpers, meta = build_swaption_helpers(base_handle, vol_shift=vol_shift)
    model = calibrate_hull_white(base_handle, helpers)
    swpt, _, _, _ = make_european_swaption(base_handle, 2, 5, 1_000_000, strike=base_atm)
    price = price_swaption_hw(swpt, model)
    a, sigma = [float(x) for x in model.params()]
    vol_rows.append({
        "scenario": label, "vol_shift": vol_shift,
        "a": a, "sigma": sigma, "NPV": price,
        "NPV_change": price - base_price,
    })
vol_sensitivity = pd.DataFrame(vol_rows)
vol_sensitivity

,scenario,vol_shift,a,sigma,NPV,NPV_change
0,-1 vol pt,-0.01,0.000012,0.005405,13838.052608,-597.071140
1,Base,0.00,0.000051,0.005639,14435.123748,0.000000
2,+1 vol pt,0.01,0.000035,0.005887,15073.028799,637.905051


## 3. Direct model-parameter bumps

This is deliberately different from recalibration: we perturb `a` or `sigma` directly while keeping the curve fixed. It helps isolate model-parameter sensitivity.

In [5]:
parameter_rows = []
parameter_cases = [
    ("Base", base_a, base_sigma),
    ("a -25%", base_a * 0.75, base_sigma),
    ("a +25%", base_a * 1.25, base_sigma),
    ("sigma -25%", base_a, base_sigma * 0.75),
    ("sigma +25%", base_a, base_sigma * 1.25),
]
for label, a, sigma in parameter_cases:
    model = ql.HullWhite(base_handle, a, sigma)
    swpt, _, _, _ = make_european_swaption(base_handle, 2, 5, 1_000_000, strike=base_atm)
    price = price_swaption_hw(swpt, model)
    parameter_rows.append({
        "scenario": label, "a": a, "sigma": sigma,
        "NPV": price, "NPV_change": price - base_price,
    })
parameter_sensitivity = pd.DataFrame(parameter_rows)
parameter_sensitivity

,scenario,a,sigma,NPV,NPV_change
0,Base,0.000051,0.005639,14435.123748,0.000000
1,a -25%,0.000038,0.005639,14435.943280,0.819531
2,a +25%,0.000063,0.005639,14434.584811,-0.538937
3,sigma -25%,0.000051,0.004229,10826.655968,-3608.467781
4,sigma +25%,0.000051,0.007048,18043.233676,3608.109928


## 4. Calibration residual review

In [6]:
base_calib[["swaption", "market_vol", "model_implied_vol", "vol_error_bp", "relative_price_error"]]

,swaption,market_vol,model_implied_vol,vol_error_bp,relative_price_error
0,1Yx1Y,0.235,0.243202,82.015049,0.034730
1,1Yx2Y,0.230,0.238398,83.979808,0.036342
2,1Yx5Y,0.223,0.225021,20.208167,0.009023
3,1Yx10Y,0.218,0.214797,-32.032851,-0.014637
4,2Yx1Y,0.232,0.234914,29.140995,0.012445
5,2Yx2Y,0.227,0.227717,7.172391,0.003132
6,2Yx5Y,0.220,0.219935,-0.645937,-0.000291
7,2Yx10Y,0.215,0.212179,-28.208952,-0.013021
8,5Yx1Y,0.224,0.213908,-100.922849,-0.044179
9,5Yx2Y,0.219,0.213855,-51.452451,-0.023043


In [7]:
calibration_summary(base_calib)

mean_abs_vol_error_bp             39.511144
max_abs_vol_error_bp             100.922849
mean_abs_relative_price_error      0.017379
dtype: float64